# Assignment — Linear Regression on `penguins.csv`

**Question:** given a penguin's measurements, how heavy is it?

Target: `body_mass_g` &nbsp;·&nbsp; File: `penguins.csv`

Use only the numeric columns: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`.

In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv('penguins.csv')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


---
## 1. Look at the data

> **Flow:** Shape, columns, missing values.

Run `.info()`. Which columns have missing values, and how many?

In [2]:
df.info()

#bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass,g,sex has missing values as 2,2,2,2,11

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


---
## 2. Clean

> **Flow:** Keep the four numeric columns, drop the rows with blanks.

We cannot fill in `body_mass_g` — that is the answer we are trying to predict.
Rows missing it have to go.

Keep `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`,
then `dropna()`.

In [ ]:
df = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
df = df.dropna()


df.head()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
4,36.7,19.3,193.0,3450.0
5,39.3,20.6,190.0,3650.0


*How many rows are left?*

In [ ]:
df.info()
#342 left


<class 'pandas.DataFrame'>
Index: 342 entries, 0 to 343
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   bill_length_mm     342 non-null    float64
 1   bill_depth_mm      342 non-null    float64
 2   flipper_length_mm  342 non-null    float64
 3   body_mass_g        342 non-null    float64
dtypes: float64(4)
memory usage: 13.4 KB


---
## 3. Features and target

In [9]:
x = df.drop(columns='body_mass_g')
y = df['body_mass_g']
print(x.shape,y.shape)


(342, 3) (342,)


---
## 4. Split

> **Flow:** `test_size=0.2`, `random_state=42`.

In [10]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
print(x_train.shape,y_train.shape)


(273, 3) (273,)


---
## 5. One feature

> **Flow:** Start with `flipper_length_mm` alone.

Train `LinearRegression`. Print the coefficient, the intercept and the R².

In [11]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train[['flipper_length_mm']], y_train)
print("coef:   ", model.coef_)
print("intercept:", model.intercept_)


coef:    [48.82968282]
intercept: -5614.067120606903


In [12]:
print(model.score(x_test[['flipper_length_mm']], y_test)) # r2 score 

0.7820354165340793


*Coefficient:* &nbsp;&nbsp; *R²:*

**Q.** The coefficient is about 48. Write one line explaining what that means in plain words —
what happens to the predicted weight if a penguin's flipper is 1 mm longer?

---
## 6. All three features

> **Flow:** Same model, two more columns.

Print the R².

In [22]:
from sklearn.linear_model import LinearRegression
model_all = LinearRegression()
model_all.fit(x_train[['bill_length_mm','bill_depth_mm','flipper_length_mm']],y_train)
print("coef:   ", model_all.coef_)
print("intercept:", model_all.intercept_)


coef:    [ 4.00878846 10.92455958 48.67371868]
intercept: -5946.0373973818


In [19]:
print(model_all.score(x_test[['bill_length_mm','bill_depth_mm','flipper_length_mm']], y_test)) # r2 score 

0.7877806019338432


*R² with flipper only:* &nbsp;&nbsp; *R² with all three:*

**Q.** The score barely moved. That is not a mistake — it is the interesting part of this
assignment. Write two lines on why adding `bill_length_mm` and `bill_depth_mm` gave almost
nothing.

*Hint: what do all three columns really measure?*

---
## 8. Metrics

> **Flow:** MAE, RMSE, R².

In [20]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model_all.predict(x_test)

print("MAE: ", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:  ", r2_score(y_test, y_pred))

MAE:  310.54744049126754
RMSE: 375.6441343110777
R2:   0.7877806019338432


*MAE:* &nbsp;&nbsp; *RMSE:* &nbsp;&nbsp; *R²:*

**Q.** The average penguin weighs about 4200 g. Is your MAE large or small compared to that?
One line.

---
## 8. Coefficients

In [21]:

# Print each feature name next to its coefficient
for feature, coefficient in zip(x.columns, model_all.coef_):
    print(feature, ":", coefficient)

bill_length_mm : 4.00878845592763
bill_depth_mm : 10.924559578456657
flipper_length_mm : 48.67371867581057


*Largest coefficient:*

**Q.** Does the largest coefficient mean that feature is the most important?
Careful — the three columns are measured on different scales. One line.

---
## 9. Questions

**Q1.** Why can we not use `accuracy_score` on this problem?

**Q2.** We dropped rows where `body_mass_g` was missing instead of filling them with the
median. Why is filling in the target a bad idea?

**Q3.** In class, adding more features to the mpg model raised R² from 0.723 to 0.824.
Here it barely changed. In two lines — what is different about these two datasets?

*Your answers:*